In [1]:
# ATTENTION: only run this cell when on google colab
!git clone https://github.com/philshiu/Drosophila_brain_model.git
!pip install brian2
%cd Drosophila_brain_model

Cloning into 'Drosophila_brain_model'...
remote: Enumerating objects: 183, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 183 (delta 75), reused 67 (delta 67), pack-reused 95 (from 1)
Receiving objects: 100% (183/183), 185.05 MiB | 34.02 MiB/s, done.
Resolving deltas: 100% (83/83), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 39.2 MB/s  0:00:00
/content/Drosophila_brain_model


In [3]:
from model import run_exp
from model import default_params as params
import utils as utl
from brian2 import Hz

config = {
    'path_res'  : './results/example',                              # directory to store results
    'path_comp' : './2023_03_23_completeness_630_final.csv',        # csv of the complete list of Flywire neurons
    'path_con'  : './2023_03_23_connectivity_630_final.parquet',    # connectivity data
    'n_proc'    : -1,                                               # number of CPU cores (-1: use all)
}

# Introduction
## Underlying connectivity data
The connectivity of the fly brain is stored in the folowing files:
- neurons present: `config['path_comp']`
- connectivity between neurons: `config['path_con]`

## Model parameters
The equation and constants for the leaky integrate and fire model are defined
in the dictionary `default_params` in the beginning of the file `model.py`:

```
default_params = {
    # trials
    't_run'     : 1000 * ms,              # duration of trial
    'n_run'     : 30,                     # number of runs

    'v_0'       : -52 * mV,               # resting potential
    'v_rst'     : -52 * mV,               # reset potential after spike
    [...]
```
We can also change values
and pass the modified dictionary to the model (see Experiment 1).

## Addressing neurons
Here, we want to stimulate some sugar-sensing neurons in the right hemisphere.
The neurons of interest are defined via their flywire IDs:

In [4]:
neu_sugar = [
    720575940624963786,
    720575940630233916,
    720575940637568838,
    720575940638202345,
    720575940617000768,
    720575940630797113,
    720575940632889389,
    720575940621754367,
    720575940621502051,
    720575940640649691,
    720575940639332736,
    720575940616885538,
    720575940639198653,
    720575940620900446,
    720575940617937543,
    720575940632425919,
    720575940633143833,
    720575940612670570,
    720575940628853239,
    720575940629176663,
    720575940611875570,
]

For an easier identification, we define also a mapping from the flywire IDs to custom
names. The above neurons are calles `sugar_1`, `sugar_2` etc:

In [5]:
flyid2name = { f: f'sugar_{i+1}' for i, f in enumerate(neu_sugar) }
flyid2name

{720575940624963786: 'sugar_1',
 720575940630233916: 'sugar_2',
 720575940637568838: 'sugar_3',
 720575940638202345: 'sugar_4',
 720575940617000768: 'sugar_5',
 720575940630797113: 'sugar_6',
 720575940632889389: 'sugar_7',
 720575940621754367: 'sugar_8',
 720575940621502051: 'sugar_9',
 720575940640649691: 'sugar_10',
 720575940639332736: 'sugar_11',
 720575940616885538: 'sugar_12',
 720575940639198653: 'sugar_13',
 720575940620900446: 'sugar_14',
 720575940617937543: 'sugar_15',
 720575940632425919: 'sugar_16',
 720575940633143833: 'sugar_17',
 720575940612670570: 'sugar_18',
 720575940628853239: 'sugar_19',
 720575940629176663: 'sugar_20',
 720575940611875570: 'sugar_21'}

# Running simulations
## Activating a set of neurons
To run a simulation exciting these nerons we have to call `run_exp` supplying the following:
- unique name for the simulation: `exp_name`
- a list of neurons we want to stimulate: `neu_sugar`
- the connectivity data: `config['path_comp']` and `config['path_con]`
- path to store the output: `config['path_res']`
- number of CPU cores use: `config['n_procs]`

Note that running this on Google Colab can take roughly 20 minutes; it is substantially faster on a local install, depending on the number of CPU cores. By default, the neurons are excited at 200 Hz.

In [6]:
# activate sugar sensing neurons
run_exp(exp_name='sugarR', neu_exc=neu_sugar, **config)

>>> Skipping experiment sugarR because results/example/sugarR.parquet exists and force_overwrite = False


The `.parquet` file created during a simulation contains all spikes events of all neurons in the model.
We load the data again from disk by passing a list of result files to the `utl.load_exps` function.

We can see from the size of the dataframe
that more than 400 000 spikes were generated by activating the sugar neurons (30 trials, 1 s each).

In [7]:
# load data from disk
df_spike = utl.load_exps([ './results/example/sugarR.parquet' ])
df_spike

,t,trial,flywire_id,exp_name
0,0.2562,0,720575940605513649,sugarR
1,0.3607,0,720575940605513649,sugarR
2,0.6044,0,720575940605513649,sugarR
3,0.7128,0,720575940605513649,sugarR
4,0.8627,0,720575940605513649,sugarR
...,...,...,...,...
511561,0.9133,29,720575940660229505,sugarR
511562,0.9245,29,720575940660229505,sugarR
511563,0.9437,29,720575940660229505,sugarR
511564,0.9687,29,720575940660229505,sugarR


The spike times can be converted to spike rates [Hz] via `utl.get_rate`, which requires the duration of each trial.
`utl.get_rate` returns `pandas.DataFrame` objects:
1. spike rate for each neuron (rows) in each experiment (column): `df_rate`
2. standard deviation of rate across trials: `df_rate_std`

For convenience, we can optionally pass the `flyid2name` dictionary to `utl.get_rate` in order to convert flywire IDs into
meaningful names.

We can see that only about 400 neurons show activity during the simulations.

In [8]:
# calculate spike rate and standard deviation
df_rate, df_rate_std = utl.get_rate(df_spike, t_run=params['t_run'], n_run=params['n_run'], flyid2name=flyid2name)
# sort by spike rate
df_rate.sort_values('sugarR', ascending=False)

exp_name,name,sugarR
flyid,,
720575940637568838,sugar_3,202.066667
720575940621502051,sugar_9,200.200000
720575940639198653,sugar_13,199.966667
720575940617937543,sugar_15,199.400000
720575940633143833,sugar_17,199.033333
...,...,...
720575940623614442,,0.033333
720575940625852464,,0.033333
720575940614465478,,0.033333


## Change stimulation frequency

We want to change the frequency of the stimulation of the sugar neurons.
To do so we modify the value for `r_poi` in the `default_params` dictionary and pass the altered dictionary to the `run_exp` function.

Note: Since physical quantities in `brian2` have to have the correct unit, we also need the `brian2.Hz` object
to define a frequency.

In [9]:
# run with different frequency
params['r_poi'] = 100 * Hz

run_exp(exp_name='sugarR_100Hz', neu_exc=neu_sugar, params=params, **config)

>>> Skipping experiment sugarR_100Hz because results/example/sugarR_100Hz.parquet exists and force_overwrite = False


We load the results via the `utl.load_exps` function and convert the spike events to rates with `utl.get_rate`

In [10]:
ps = [
    './results/example/sugarR.parquet',
    './results/example/sugarR_100Hz.parquet',
]

df_spike = utl.load_exps(ps)
df_rate, df_rate_std = utl.get_rate(df_spike, t_run=params['t_run'], n_run=params['n_run'], flyid2name=flyid2name)
df_rate.sort_values('sugarR_100Hz', ascending=False, inplace=True)
df_rate

exp_name,name,sugarR,sugarR_100Hz
flyid,,,
720575940622695448,,156.766667,114.266667
720575940621754367,sugar_8,194.100000,102.566667
720575940617937543,sugar_15,199.400000,101.733333
720575940629888530,,146.500000,101.700000
720575940638202345,sugar_4,191.266667,101.266667
...,...,...,...
720575940639813365,,0.066667,NaN
720575940643551392,,0.733333,NaN
720575940644666148,,1.133333,NaN


## Silencing neurons
We want to silence the most active neurons individually to see how that changes the activity patterns.
We do so by passing the neuron IDs we want to silence as a list `run_exp` via the `neu_slnc` argument.
In the following example, we are silencing a single neuron `[ i ]` while exciting the sugar neurons `neu_sugar`.
We can then investigate how silencing of each individual neuron affects the firing rate of a given neuron, say, MN9.

In [11]:
#First, let's check on the MN9 firing rate when no neurons are silenced.
id_mn9 = 720575940660219265 #id for MN9
x = df_rate.loc[id_mn9, "sugarR_100Hz"]
print(f'Rate for neuron {id_mn9} is {x}')

Rate for neuron 720575940660219265 is 67.03333333333333


In [12]:
# IDs of 3 most active neurons. These neurons are all sugar-sensing neurons.
ids = df_rate.sort_values('sugarR_100Hz', ascending=False).index[:3]

for i in ids:
    run_exp(exp_name=f'sugarR-{i}', neu_exc=neu_sugar, neu_slnc=[ i ], params=params, **config)

>>> Skipping experiment sugarR-720575940622695448 because results/example/sugarR-720575940622695448.parquet exists and force_overwrite = False
>>> Skipping experiment sugarR-720575940621754367 because results/example/sugarR-720575940621754367.parquet exists and force_overwrite = False
>>> Skipping experiment sugarR-720575940617937543 because results/example/sugarR-720575940617937543.parquet exists and force_overwrite = False


In [13]:
# output files
ps = [ f'./results/example/sugarR-{i}.parquet' for i in ids ]

# calculate spike rate and sort
df_spike = utl.load_exps(ps)
df_rate, df_rate_std = utl.get_rate(df_spike, t_run=params['t_run'], n_run=params['n_run'])
df_rate.loc[id_mn9, :].sort_values(ascending=True)

,720575940660219265
exp_name,
sugarR-720575940617937543,63.266667
sugarR-720575940621754367,66.900000
sugarR-720575940622695448,71.166667


Load the cell type neuron lists. This shows how the names are spelled.

In [14]:
import pickle
with open('sez_neurons.pickle', 'rb') as f:
    sez = pickle.load(f)
print([k for k in sez if k.lower() in ('roundup', 'usnea')])

['roundup', 'usnea']


Run roundup at 50 Hz (quick test with 5 runs).
Use whatever spelling Cell 1 printed. %%time reports how long the run took. The new exp_name means nothing gets skipped this time.

In [17]:
%%time
params['r_poi'] = 50 * Hz
params['n_run'] = 5          # quick test; the paper used 30
run_exp(exp_name='usnea_50Hz_test', neu_exc=sez['usnea'], params=params, **config)

>>> Experiment:     usnea_50Hz_test
    Output file:    results/example/usnea_50Hz_test.parquet
    Excited neurons: 2
    Elapsed time:   71 s
CPU times: user 409 ms, sys: 65.8 ms, total: 475 ms
Wall time: 1min 11s


Read the left MN9 rate.
The reindex(...).fillna(0) part handles the case where MN9 never fired: you get 0 instead of an error. You're aiming for roughly 70–90 Hz for roundup. Then repeat Cells 2 and 3 with usnea, changing both the name and the exp_name (e.g. usnea_50Hz_test), and you should get about 0.

In [18]:
p = f"{config['path_res']}/usnea_50Hz_test.parquet"
df_spike = utl.load_exps([p])
df_rate, df_rate_std = utl.get_rate(df_spike, t_run=params['t_run'], n_run=params['n_run'])
print(df_rate.reindex([id_mn9]).fillna(0))

exp_name            usnea_50Hz_test
flyid                              
720575940660219265              0.0


Connect Drive.

In [19]:
from google.colab import drive
drive.mount('/content/drive')

import os
config['path_res'] = '/content/drive/MyDrive/fly_project/results_v630'
os.makedirs(config['path_res'], exist_ok=True)

Mounted at /content/drive


Run the batch.

In [22]:
key_types = ['roundup', 'diatom', 'sink_sync', 'G2N_1', 'clavicle', 'Fdg', 'bract',
             'rattle', 'FMIn', 'TH_VUM', 'marge', 'phantom', 'tentacular', 'usnea',
             'kitty', 'peacock', 'peahen']
lookup = {k.casefold(): k for k in sez}   # handles capitalisation differences

params['r_poi'] = 50 * Hz
params['n_run'] = 30
for name in key_types:
    run_exp(exp_name=f'{name}_50Hz', neu_exc=sez[lookup[name.casefold()]],
            params=params, **config)

>>> Experiment:     roundup_50Hz
    Output file:    /content/drive/MyDrive/fly_project/results_v630/roundup_50Hz.parquet
    Excited neurons: 2
    Elapsed time:   457 s
>>> Experiment:     diatom_50Hz
    Output file:    /content/drive/MyDrive/fly_project/results_v630/diatom_50Hz.parquet
    Excited neurons: 14


WARNING    /usr/local/lib/python3.13/dist-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
 [py.warnings]
  warnings.warn(



    Elapsed time:   1005 s
>>> Experiment:     sink_sync_50Hz
    Output file:    /content/drive/MyDrive/fly_project/results_v630/sink_sync_50Hz.parquet
    Excited neurons: 4


WARNING    /usr/local/lib/python3.13/dist-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
 [py.warnings]
  warnings.warn(



    Elapsed time:   543 s
>>> Experiment:     G2N_1_50Hz
    Output file:    /content/drive/MyDrive/fly_project/results_v630/G2N_1_50Hz.parquet
    Excited neurons: 2


WARNING    /usr/local/lib/python3.13/dist-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
 [py.warnings]
  warnings.warn(



... (11 outputs left out)

WARNING    /usr/local/lib/python3.13/dist-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
 [py.warnings]
  warnings.warn(



    Elapsed time:   505 s
>>> Experiment:     peacock_50Hz
    Output file:    /content/drive/MyDrive/fly_project/results_v630/peacock_50Hz.parquet
    Excited neurons: 4
    Elapsed time:   413 s
>>> Experiment:     peahen_50Hz
    Output file:    /content/drive/MyDrive/fly_project/results_v630/peahen_50Hz.parquet
    Excited neurons: 4


WARNING    /usr/local/lib/python3.13/dist-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
 [py.warnings]
  warnings.warn(



    Elapsed time:   438 s


In [23]:
import os
import pandas as pd

# 1. Check every file exists and every experiment has all 30 runs
ps = [f"{config['path_res']}/{n}_50Hz.parquet" for n in key_types]
print(sum(os.path.exists(p) for p in ps), "of", len(ps), "files saved")
df_spike = utl.load_exps(ps)
print(df_spike.groupby('exp_name')['trial'].nunique())

# 2. Your MN9 rates vs the paper's (left_50 from Supplementary Table 3)
df_rate, df_rate_std = utl.get_rate(df_spike, t_run=params['t_run'], n_run=params['n_run'])
id_mn9_right = 720575940645521262
published = {'roundup': 79.5, 'diatom': 29.133, 'sink_sync': 22.167, 'G2N_1': 12.367,
             'clavicle': 10.5, 'Fdg': 22.533, 'bract': 33.667, 'rattle': 1.867,
             'FMIn': 0.367, 'TH_VUM': 0.033, 'marge': 0, 'phantom': 0,
             'tentacular': 0, 'usnea': 0, 'kitty': 7.3, 'peacock': 0, 'peahen': 0}

cols = [f'{n}_50Hz' for n in key_types]
left = df_rate.reindex(index=[id_mn9], columns=cols).fillna(0).iloc[0].values
left_sd = df_rate_std.reindex(index=[id_mn9], columns=cols).fillna(0).iloc[0].values
right = df_rate.reindex(index=[id_mn9_right], columns=cols).fillna(0).iloc[0].values

table = pd.DataFrame({
    'cell_type': key_types,
    'published_left': [published[n] for n in key_types],
    'my_left': left.round(2),
    'my_left_sd': left_sd.round(2),
    'my_right': right.round(3),
})
table['paper_says_move'] = table['published_left'] > 0
table['i_say_move'] = table['my_left'] > 0
table['agree'] = table['paper_says_move'] == table['i_say_move']
table.to_csv(f"{config['path_res']}/v630_reproduction_17.csv", index=False)
table

17 of 17 files saved
exp_name
FMIn_50Hz          30
Fdg_50Hz           30
G2N_1_50Hz         30
TH_VUM_50Hz        30
bract_50Hz         30
clavicle_50Hz      30
diatom_50Hz        30
kitty_50Hz         30
marge_50Hz         30
peacock_50Hz       30
peahen_50Hz        30
phantom_50Hz       30
rattle_50Hz        30
roundup_50Hz       30
sink_sync_50Hz     30
tentacular_50Hz    30
usnea_50Hz         30
Name: trial, dtype: int64


,cell_type,published_left,my_left,my_left_sd,my_right,paper_says_move,i_say_move,agree
0,roundup,79.500,78.77,7.33,82.267,True,True,True
1,diatom,29.133,30.47,33.07,30.733,True,True,True
2,sink_sync,22.167,26.67,9.11,27.600,True,True,True
3,G2N_1,12.367,9.17,4.78,9.733,True,True,True
4,clavicle,10.500,10.00,2.62,6.733,True,True,True
5,Fdg,22.533,22.90,4.91,20.933,True,True,True
6,bract,33.667,32.60,4.85,24.233,True,True,True
7,rattle,1.867,2.57,2.14,1.367,True,True,True
8,FMIn,0.367,0.60,1.02,0.167,True,True,True
9,TH_VUM,0.033,0.00,0.00,2.600,True,False,False


In [24]:
from google.colab import files
files.upload_file("v630_batch_check.py")

from v630_batch_check import check_batch
summary_csv = check_batch(config, params)
files.download(str(summary_csv))

Saved v630_batch_check.py to /content/Drosophila_brain_model/v630_batch_check.py
Reading /content/drive/MyDrive/fly_project/results_v630 | 30 trials of 1 s | 50 Hz input
The current model inputs are 2023_03_23_completeness_630_final.csv and 2023_03_23_connectivity_630_final.parquet

17/17 files contain spike records for all 30 trials.
 cell_type status  trials_seen  left_hz  paper_left_hz  right_hz  paper_right_hz  left_matches_paper_call left_vs_lab
   roundup     ok           30  78.7667        79.5000   82.2667         82.6667                     True          TP
    diatom     ok           30  30.4667        29.1333   30.7333         28.3333                     True          TP
 sink_sync     ok           30  26.6667        22.1667   27.6000         22.7667                     True          TP
     G2N_1     ok           30   9.1667        12.3667    9.7333         13.9667                     True          TP
  clavicle     ok           30  10.0000        10.5000    6.7333         

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
import sys, brian2
print("Python", sys.version.split()[0])
print("Brian2", brian2.__version__)
!git -C /content/Drosophila_brain_model rev-parse --short HEAD

Python 3.13.18
Brian2 2.11.0
91bdd1e
